# 💬 Prompt Messages in LangChain

## Learning Objectives
In this notebook, you will learn:
1. **ChatPromptTemplate Basics** - How to build a prompt template with named `{placeholder}` variables
2. **Multi-Message Templates** - How to combine system and human messages into a single reusable template
3. **Message Types** - The core message classes LangChain uses to represent a conversation
4. **Few-Shot Prompting** - How to steer a model's output using example input/output pairs
5. **Reusable Prompt Components** - How to compose prompts from smaller, reusable pieces with `+`

## Prerequisites
- Basic understanding of LangChain (see `01_core_concepts.ipynb`)
- Familiarity with LLM initialization (see `02_working_with_llms.ipynb`)
- `OPENAI_API_KEY` set in a `.env` file at the project root

---
## 🔧 Setup

We load environment variables from `.env` and import the LangChain building blocks used throughout this notebook: `ChatPromptTemplate` and `FewShotChatMessagePromptTemplate` for building prompts, the core message classes for representing conversation turns, and `init_chat_model` for creating a chat model.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and API Keys
# ============================================================================
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, ChatMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

load_dotenv()

print("✅ Environment loaded and LangChain imports ready!")

### 1. 📝 Basic ChatPromptTemplate

`ChatPromptTemplate.from_template` builds a single-message prompt from a plain string containing `{placeholder}` variables. Calling `format_messages` fills in those placeholders and returns a list of LangChain message objects ready to send to a model.

In [ ]:
# ============================================================================
# BASIC CHATPROMPTTEMPLATE: Single-Placeholder Template
# ============================================================================
prompt = ChatPromptTemplate.from_template("Tell me a {adjective} joke about {topic}.")

# Fill in the placeholders and inspect the resulting message list
messages = prompt.format_messages(adjective="funny", topic="chickens")

print(messages)

### 2. 💬 Multi-Message Templates

`ChatPromptTemplate.from_messages` builds a multi-turn template from a list of `(role, text)` tuples (e.g. `"system"`, `"human"`). This is the standard way to define both the assistant's behavior and the user's request in one reusable template.

In [ ]:
# ============================================================================
# MULTI-MESSAGE TEMPLATES: Combining System and Human Roles
# ============================================================================
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant that translates {input_language} to {output_language}.",
        ),
        ("human", "Translate the following text: {text}"),
    ]
)

messages = prompt.format_messages(
    input_language="English", output_language="French", text="I love programming."
)

print(messages)

### 3. 📨 Message Types

LangChain represents each turn of a conversation as a typed message object rather than a plain string. `HumanMessage`, `AIMessage`, `SystemMessage`, `ToolMessage`, and `ChatMessage` cover the roles a model or tool can play in a chat history.

In [ ]:
# ============================================================================
# MESSAGE TYPES: The Building Blocks of a Conversation
# ============================================================================
messages = [
    HumanMessage(content="Hello!"),
    AIMessage(content="Hi there! How can I assist you today?"),
    SystemMessage(content="This is a system message."),
    ToolMessage(content="Tool executed successfully.", tool_call_id="call_123"),
    ChatMessage(content="This is a general chat message."),
]

for message in messages:
    print(f"{type(message).__name__}: {message.content}")

### 4. 🎯 Few-Shot Prompting

`FewShotChatMessagePromptTemplate` shows the model a handful of example input/output pairs before the real question, steering its response format through demonstration rather than instruction alone. Here we teach the model to output the opposite of a given word.

In [ ]:
# ============================================================================
# FEW-SHOT PROMPTING: Building the Prompt from Example Pairs
# ============================================================================
examples = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
]

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

fewshot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Give the opposite of each word."),
        fewshot_prompt,
        ("human", "{input}"),
    ]
)

print(final_prompt.format_messages(input="happy"))

In [ ]:
# ============================================================================
# FEW-SHOT PROMPTING: Running the Prompt Through a Model
# ============================================================================
model = init_chat_model(model="gpt-4o-mini", temperature=0)
response = model.invoke(final_prompt.format_messages(input="happy"))

print(response.content)

### 5. 🧩 Reusable Prompt Components

`ChatPromptTemplate` objects support the `+` operator, letting you compose a final prompt out of smaller, independently reusable pieces - for example a system-role template and a human-role template defined separately.

In [ ]:
# ============================================================================
# REUSABLE COMPONENTS: Composing Prompts with the + Operator
# ============================================================================
system_prompt = ChatPromptTemplate.from_messages([("system", "You are a {role}.")])
user_prompt = ChatPromptTemplate.from_messages([("human", "{question}")])

# Combine the two templates into a single reusable prompt
full_prompt = system_prompt + user_prompt

messages = full_prompt.format_messages(role="helpful assistant", question="What is AI?")

print(messages)

---
## 📝 Summary

In this notebook, we learned:

### 1. ChatPromptTemplate Basics
- **`from_template`**: Build a single-message prompt from a string with `{placeholder}` variables
- **`format_messages`**: Fill in the placeholders and get back a list of LangChain message objects

### 2. Multi-Message and Reusable Templates
- **`from_messages`**: Build multi-turn templates from `(role, text)` tuples (e.g. `"system"`, `"human"`)
- **Composition**: `ChatPromptTemplate` objects can be combined with `+` to build larger prompts from smaller, reusable pieces

### 3. Message Types
- **`HumanMessage` / `AIMessage` / `SystemMessage` / `ToolMessage` / `ChatMessage`**: The typed objects LangChain uses to represent each turn of a conversation

### 4. Few-Shot Prompting
- **`FewShotChatMessagePromptTemplate`**: Show the model example input/output pairs before the real question, steering its output format through demonstration rather than instruction alone

### Next Steps
- Continue to `04_prompt_templates_all.ipynb` to explore the full range of prompt template types